# Imputacion valores NAN y OUTLIERS segun rango temporal con metodo por Transecto y metodo General

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # Imputacion valores NAN y OUTLIERS segun rango temporal con metodo por Transecto y metodo General
# Versión corregida: maneja índices duplicados al imputar gaps y evita errores de reindexación.
# Cambios principales:
#  - Lectura de CSV con low_memory=False para evitar DtypeWarning.
#  - Imputación por horas y asignaciones realizadas por posición (i.e. iat/iloc) para soportar índices duplicados.
#  - Mensajes de depuración mínimos para gaps no encontrados en el índice.
#
# Entrada: Archivos *_outliers.csv en INPUT_DIR
# Salida:
#  - imputed_by_transect/: archivos Transecto_1.csv, Transecto_2.csv, ...
#  - imputed_global/     : archivos por estación

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import KNNImputer

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/usu/snsaetor/Documents/GitHub/TFGFinal/Datos_TFG_outliers/")
INPUT_DIR = BASE_DIR  # donde están los *_outliers.csv

OUTPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
OUTPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_TRANSECT, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

# Umbrales (horas)
SHORT_GAP_MAX = 6
MEDIUM_GAP_MAX = 72
LONG_GAP_MIN = 72

# Método para imputación estacional ('mean' o 'median')
SEASONAL_STAT = 'mean'

# Columnas numéricas a imputar (incluye tanto O3_for_impute como O3 por seguridad)
NUM_COLS = [
    'NO', 'NO2', 'NOx', 'O3_for_impute', 'O3',
    'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Dist.', 'Angulo'
]

# Para KNN
KNN_NEIGHBORS = 5

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def load_station_data(filepath):
    """Carga un CSV con índice datetime."""
    # low_memory=False para evitar DtypeWarning en columnas mezcladas
    df = pd.read_csv(filepath, index_col=0, parse_dates=True, low_memory=False)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    return df

def get_gap_lengths(series):
    """
    Devuelve un dict {tuple(indice): longitud} para cada grupo consecutivo de NaN en 'series'.
    Cada key es una tupla de timestamps (puede contener timestamps repetidos si existían en el índice).
    """
    mask = series.isna()
    if mask.empty:
        return {}
    groups = mask.ne(mask.shift()).cumsum()
    gap_groups = {}
    # iterar sobre grupos donde mask == True
    for g in groups[mask].unique():
        idx = groups[groups == g].index
        gap_groups[tuple(idx)] = len(idx)
    return gap_groups

def compute_hourly_stats(series, stat='mean'):
    """
    Calcula el estadístico horario (media o mediana) y asegura que existan valores para todas las horas (0-23).
    Si no hay datos para una hora, se rellena con el valor más cercano (ffill/bfill).
    Si la serie está vacía, retorna Serie de NaN.
    """
    if series is None or series.empty:
        return pd.Series(np.nan, index=range(24))
    if stat == 'mean':
        hourly = series.groupby(series.index.hour).mean()
    else:
        hourly = series.groupby(series.index.hour).median()
    hourly = hourly.reindex(range(24))
    hourly = hourly.ffill().bfill()
    return hourly

def impute_dataframe(df, hourly_stats_dict):
    """
    Imputa un DataFrame de una estación:
      - Gaps cortos (<= SHORT_GAP_MAX): interpolación lineal.
      - Gaps medios (< MEDIUM_GAP_MAX): imputación por estadístico horario.
      - Gaps largos (>= MEDIUM_GAP_MAX): se dejan para KNN.
    Esta versión maneja índices duplicados asignando por posición.
    """
    df_imp = df.copy()
    # Precalcular un array del índice para comparaciones rápidas
    index_values = df_imp.index.values

    for col in NUM_COLS:
        if col not in df_imp.columns:
            continue
        # Forzar tipo numérico
        df_imp[col] = pd.to_numeric(df_imp[col], errors='coerce')
        serie = df_imp[col]
        if serie.isna().sum() == 0:
            continue

        gaps = get_gap_lengths(serie)
        if not gaps:
            continue

        # ubicación de la columna en df_imp (entera) para uso con iat
        col_loc = df_imp.columns.get_loc(col)

        for idx_tuple, L in gaps.items():
            # idx contiene los timestamps (posiblemente repetidos) del gap
            idx = pd.Index(idx_tuple)

            if L <= SHORT_GAP_MAX:
                # interpolación lineal global en la serie (no solo en el gap)
                serie_interp = serie.interpolate(method='linear', limit_direction='both')
                # Para evitar reindex problemas con duplicados, asignamos por posiciones encontradas para cada timestamp
                for ts in idx:
                    # obtener posiciones donde el índice coincide exactamente con ts
                    positions = np.where(index_values == np.datetime64(ts))[0]
                    if positions.size == 0:
                        # timestamp no encontrado en el índice (raro), lo ignoramos
                        # (opcional: registrar)
                        # print(f"  Nota: timestamp {ts} del gap no se encontró en el índice de estación.")
                        continue
                    # valor interpolado en esa fecha/posición (tomamos la primera posición para leer el valor)
                    # si la posición existe en serie_interp, usamos iloc para tomar el valor en la posición encontrada
                    # (si hay varias filas con mismo timestamp, asignaremos el mismo valor a todas ellas)
                    val = None
                    try:
                        val = serie_interp.iloc[positions[0]]
                    except Exception:
                        # fallback: intentar seleccionar por etiqueta (puede devolver array si hay duplicados)
                        try:
                            val = serie_interp.loc[ts]
                            # si loc devolvió una Serie (duplicados), tomar el primer elemento
                            if isinstance(val, (pd.Series, np.ndarray, list)):
                                val = val.iloc[0] if isinstance(val, pd.Series) else val[0]
                        except Exception:
                            val = np.nan
                    # asignar a todas las posiciones correspondientes
                    for p in positions:
                        df_imp.iat[p, col_loc] = val

            elif L < MEDIUM_GAP_MAX:
                # imputación por estadístico horario si existe
                if col in hourly_stats_dict:
                    hourly_series = hourly_stats_dict[col]  # Serie con índice 0..23
                    for ts in idx:
                        positions = np.where(index_values == np.datetime64(ts))[0]
                        if positions.size == 0:
                            # timestamp no presente en el DataFrame (ignorar)
                            continue
                        # calcular hora
                        try:
                            hour = pd.to_datetime(ts).hour
                        except Exception:
                            # si no se puede convertir, saltar
                            continue
                        # obtener valor horario (si no existe, NaN)
                        try:
                            # hourly_series es una Series con índices 0..23
                            val = hourly_series.loc[hour]
                        except Exception:
                            # si falla, intentar get
                            val = hourly_series.get(hour, np.nan) if hasattr(hourly_series, 'get') else np.nan
                        # asignar a todas las posiciones correspondientes
                        for p in positions:
                            df_imp.iat[p, col_loc] = val
                else:
                    # no hay estadístico horario para esta columna -> dejamos NaN para KNN
                    pass
            else:
                # gap largo: dejamos NaN para KNN
                pass

    return df_imp

def apply_knn_imputation(df, num_cols, extra_cols=None):
    """
    Aplica KNNImputer a las columnas num_cols (solo las que existan en df),
    usando extra_cols como predictores (deben ser numéricos).
    Retorna df con las columnas imputadas.
    """
    cols_to_impute = [c for c in num_cols if c in df.columns]
    if not cols_to_impute:
        return df

    if extra_cols:
        extra_cols = [c for c in extra_cols if c in df.columns]
        all_cols = cols_to_impute + extra_cols
    else:
        all_cols = cols_to_impute

    sub = df[all_cols].copy()
    for c in sub.columns:
        sub[c] = pd.to_numeric(sub[c], errors='coerce')

    if extra_cols:
        for col in extra_cols:
            if sub[col].isna().any():
                sub[col].fillna(-999, inplace=True)

    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS)
    try:
        imputed_array = imputer.fit_transform(sub)
    except Exception as e:
        print("  ERROR en KNNImputer:", e)
        print("  Intentando imputación de fallback por media de columna.")
        for c in cols_to_impute:
            mean_val = sub[c].mean(skipna=True)
            sub[c].fillna(mean_val if not pd.isna(mean_val) else 0.0, inplace=True)
        imputed_array = sub.values

    df_imp = df.copy()
    df_imp.loc[:, cols_to_impute] = imputed_array[:, :len(cols_to_impute)]
    return df_imp

def fill_remaining_nans_with_mean(df, cols):
    """
    Rellena los NaN que aún puedan quedar en las columnas especificadas con la media de cada columna.
    Si una columna es completamente NaN, se rellena con 0 (último recurso).
    """
    for col in cols:
        if col in df.columns and df[col].isna().any():
            mean_val = pd.to_numeric(df[col], errors='coerce').mean(skipna=True)
            if pd.isna(mean_val):
                mean_val = 0.0
            df[col] = df[col].fillna(mean_val)
    return df

def ensure_no_nans(df, cols):
    """
    Verifica que no queden NaN en cols; si quedan, rellena con 0 e imprime advertencia.
    """
    for col in cols:
        if col in df.columns and df[col].isna().any():
            n_nans = int(df[col].isna().sum())
            print(f"  ADVERTENCIA: {n_nans} NaN residuales en '{col}' rellenados con 0.")
            df[col] = df[col].fillna(0)
    return df

# ============================================================================
# PROCESAMIENTO POR TRANSECTO (MEJORADO)
# ============================================================================

def process_by_transect():
    print("\n=== Imputación por transecto (agrupando estaciones) ===")

    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    transect_dict = {}

    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        df = load_station_data(filepath)

        if 'O3' in df.columns and 'O3_for_impute' not in df.columns:
            df['O3_for_impute'] = pd.to_numeric(df['O3'], errors='coerce')

        if 'Estacion' not in df.columns:
            df['Estacion'] = station_name
        else:
            df['Estacion'] = df['Estacion'].fillna(station_name)

        if 'Transecto' not in df.columns:
            print(f"  Advertencia: {filepath.name} no tiene columna 'Transecto'. Se omite.")
            continue
        transect_names = df['Transecto'].dropna().unique()
        if len(transect_names) == 0:
            print(f"  Advertencia: {filepath.name} no tiene valores en Transecto. Se omite.")
            continue
        transect = transect_names[0]
        df['Transecto'] = df['Transecto'].fillna(transect)

        transect_clean = str(transect).replace(" ", "_")

        if transect_clean not in transect_dict:
            transect_dict[transect_clean] = []
        transect_dict[transect_clean].append((station_name, df))

    for transect_clean, station_list in transect_dict.items():
        print(f"\nProcesando transecto: {transect_clean}")

        df_all = pd.concat([df for _, df in station_list], axis=0, sort=False)
        df_all = df_all.sort_index()

        hourly_stats = {}
        for col in NUM_COLS:
            if col not in df_all.columns:
                continue
            s = pd.to_numeric(df_all[col], errors='coerce').dropna()
            hourly_stats[col] = compute_hourly_stats(s, SEASONAL_STAT)

        df_imputed_list = []
        for station_name, df_station in station_list:
            df_station['Estacion'] = df_station['Estacion'].fillna(station_name)
            if 'Transecto' in df_station.columns:
                df_station['Transecto'] = df_station['Transecto'].fillna(transect.replace("_", " "))
            for col in NUM_COLS:
                if col in df_station.columns:
                    df_station[col] = pd.to_numeric(df_station[col], errors='coerce')

            df_station_imp = impute_dataframe(df_station, hourly_stats)
            df_imputed_list.append(df_station_imp)

        df_concat = pd.concat(df_imputed_list, axis=0, sort=False)
        df_concat = df_concat.sort_index()

        # Reemplazamos fillna(method=...) por ffill().bfill()
        if 'Estacion' in df_concat.columns:
            df_concat['Estacion'] = df_concat['Estacion'].ffill().bfill().fillna('unknown_station')
        if 'Transecto' in df_concat.columns:
            df_concat['Transecto'] = df_concat['Transecto'].ffill().bfill().fillna(transect_clean.replace("_", " "))

        df_concat['station_code'] = pd.factorize(df_concat['Estacion'])[0]
        extra_cols = ['station_code']

        df_concat = apply_knn_imputation(df_concat, NUM_COLS, extra_cols=extra_cols)
        df_concat = fill_remaining_nans_with_mean(df_concat, NUM_COLS)
        df_concat = ensure_no_nans(df_concat, NUM_COLS)

        if 'station_code' in df_concat.columns:
            df_concat.drop(columns=['station_code'], inplace=True)

        if 'O3_for_impute' in df_concat.columns:
            df_concat['O3'] = df_concat['O3_for_impute']
            df_concat.drop(columns=['O3_for_impute'], inplace=True)

        out_path = os.path.join(OUTPUT_BY_TRANSECT, f"{transect_clean}.csv")
        df_concat.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path} (shape: {df_concat.shape})")

# ============================================================================
# PROCESAMIENTO GLOBAL (MEJORADO)
# ============================================================================

def process_global():
    print("\n=== Imputación global (todas las estaciones) ===")
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    dfs = []
    station_names = []
    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        station_names.append(station_name)
        df = load_station_data(filepath)

        if 'O3' in df.columns and 'O3_for_impute' not in df.columns:
            df['O3_for_impute'] = pd.to_numeric(df['O3'], errors='coerce')

        if 'Estacion' not in df.columns:
            df['Estacion'] = station_name
        else:
            df['Estacion'] = df['Estacion'].fillna(station_name)

        if 'Transecto' in df.columns:
            tran_names = df['Transecto'].dropna().unique()
            if len(tran_names) > 0:
                df['Transecto'] = df['Transecto'].fillna(tran_names[0])

        df['station_id'] = station_name
        dfs.append(df)

    df_all = pd.concat(dfs, axis=0, sort=False)
    df_all = df_all.sort_index()

    global_hourly_stats = {}
    for col in NUM_COLS:
        if col not in df_all.columns:
            continue
        s = pd.to_numeric(df_all[col], errors='coerce').dropna()
        global_hourly_stats[col] = compute_hourly_stats(s, SEASONAL_STAT)

    df_all_imp = df_all.copy()
    for station in station_names:
        mask = df_all_imp['station_id'] == station
        df_station = df_all_imp.loc[mask].copy()
        for col in NUM_COLS:
            if col in df_station.columns:
                df_station[col] = pd.to_numeric(df_station[col], errors='coerce')
        df_station = impute_dataframe(df_station, global_hourly_stats)
        df_all_imp.loc[mask] = df_station

    df_all_imp['station_code'] = pd.factorize(df_all_imp['station_id'])[0]
    extra_cols = ['station_code']
    df_all_imp = apply_knn_imputation(df_all_imp, NUM_COLS, extra_cols=extra_cols)
    df_all_imp = fill_remaining_nans_with_mean(df_all_imp, NUM_COLS)
    df_all_imp = ensure_no_nans(df_all_imp, NUM_COLS)

    if 'station_code' in df_all_imp.columns:
        df_all_imp.drop(columns=['station_code'], inplace=True)

    if 'O3_for_impute' in df_all_imp.columns:
        df_all_imp['O3'] = df_all_imp['O3_for_impute']
        df_all_imp.drop(columns=['O3_for_impute'], inplace=True)

    for station in station_names:
        df_station = df_all_imp[df_all_imp['station_id'] == station].copy()
        if 'station_id' in df_station.columns:
            df_station.drop(columns=['station_id'], inplace=True)
        if 'Estacion' not in df_station.columns or df_station['Estacion'].isna().any():
            df_station['Estacion'] = df_station['Estacion'].ffill().bfill().fillna(station)
        if 'Transecto' in df_station.columns:
            df_station['Transecto'] = df_station['Transecto'].ffill().bfill().fillna('unknown_transect')

        out_path = os.path.join(OUTPUT_GLOBAL, f"{station}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path}")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("Iniciando imputación de datos (por transecto y global)...")
    process_by_transect()
    process_global()
    print("\nProceso completado. Revise las carpetas:")
    print(f"  - {OUTPUT_BY_TRANSECT}")
    print(f"  - {OUTPUT_GLOBAL}")

# Optimizado

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # Imputacion valores NAN y OUTLIERS segun rango temporal con metodo por Transecto y metodo General
# Versión corregida: maneja índices duplicados al imputar gaps y evita errores de reindexación.
# Cambios principales:
#  - Lectura de CSV con low_memory=False para evitar DtypeWarning.
#  - Imputación por horas y asignaciones realizadas por posición (i.e. iat/iloc) para soportar índices duplicados.
#  - Mensajes de depuración mínimos para gaps no encontrados en el índice.
#
# Entrada: Archivos *_outliers.csv en INPUT_DIR
# Salida:
#  - imputed_by_transect/: archivos Transecto_1.csv, Transecto_2.csv, ...
#  - imputed_global/     : archivos por estación

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import KNNImputer

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/usu/snsaetor/Documents/GitHub/TFGFinal/Datos_TFG_outliers/")
INPUT_DIR = BASE_DIR  # donde están los *_outliers.csv

OUTPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
OUTPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_TRANSECT, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

# Umbrales (horas)
SHORT_GAP_MAX = 6
MEDIUM_GAP_MAX = 72
LONG_GAP_MIN = 72

# Método para imputación estacional ('mean' o 'median')
SEASONAL_STAT = 'mean'

# Columnas numéricas a imputar (incluye tanto O3_for_impute como O3 por seguridad)
NUM_COLS = [
    'NO', 'NO2', 'NOx', 'O3_for_impute', 'O3',
    'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Dist.', 'Angulo'
]

# Para KNN
KNN_NEIGHBORS = 5

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def load_station_data(filepath):
    """Carga un CSV con índice datetime."""
    # low_memory=False para evitar DtypeWarning en columnas mezcladas
    df = pd.read_csv(filepath, index_col=0, parse_dates=True, low_memory=False)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    return df

def get_gap_lengths(series):
    """
    Devuelve un dict {tuple(indice): longitud} para cada grupo consecutivo de NaN en 'series'.
    Cada key es una tupla de timestamps (puede contener timestamps repetidos si existían en el índice).
    """
    mask = series.isna()
    if mask.empty:
        return {}
    groups = mask.ne(mask.shift()).cumsum()
    gap_groups = {}
    # iterar sobre grupos donde mask == True
    for g in groups[mask].unique():
        idx = groups[groups == g].index
        gap_groups[tuple(idx)] = len(idx)
    return gap_groups

def compute_hourly_stats(series, stat='mean'):
    """
    Calcula el estadístico horario (media o mediana) y asegura que existan valores para todas las horas (0-23).
    Si no hay datos para una hora, se rellena con el valor más cercano (ffill/bfill).
    Si la serie está vacía, retorna Serie de NaN.
    """
    if series is None or series.empty:
        return pd.Series(np.nan, index=range(24))
    if stat == 'mean':
        hourly = series.groupby(series.index.hour).mean()
    else:
        hourly = series.groupby(series.index.hour).median()
    hourly = hourly.reindex(range(24))
    hourly = hourly.ffill().bfill()
    return hourly

def impute_dataframe(df, hourly_stats_dict):
    """
    Imputa un DataFrame de una estación:
      - Gaps cortos (<= SHORT_GAP_MAX): interpolación lineal.
      - Gaps medios (< MEDIUM_GAP_MAX): imputación por estadístico horario.
      - Gaps largos (>= MEDIUM_GAP_MAX): se dejan para KNN.
    Esta versión maneja índices duplicados asignando por posición.
    """
    df_imp = df.copy()
    # Precalcular un array del índice para comparaciones rápidas
    index_values = df_imp.index.values

    for col in NUM_COLS:
        if col not in df_imp.columns:
            continue
        # Forzar tipo numérico
        df_imp[col] = pd.to_numeric(df_imp[col], errors='coerce')
        serie = df_imp[col]
        if serie.isna().sum() == 0:
            continue

        gaps = get_gap_lengths(serie)
        if not gaps:
            continue

        # ubicación de la columna en df_imp (entera) para uso con iat
        col_loc = df_imp.columns.get_loc(col)

        for idx_tuple, L in gaps.items():
            # idx contiene los timestamps (posiblemente repetidos) del gap
            idx = pd.Index(idx_tuple)

            if L <= SHORT_GAP_MAX:
                # interpolación lineal global en la serie (no solo en el gap)
                serie_interp = serie.interpolate(method='linear', limit_direction='both')
                # Para evitar reindex problemas con duplicados, asignamos por posiciones encontradas para cada timestamp
                for ts in idx:
                    # obtener posiciones donde el índice coincide exactamente con ts
                    positions = np.where(index_values == np.datetime64(ts))[0]
                    if positions.size == 0:
                        # timestamp no encontrado en el índice (raro), lo ignoramos
                        # (opcional: registrar)
                        # print(f"  Nota: timestamp {ts} del gap no se encontró en el índice de estación.")
                        continue
                    # valor interpolado en esa fecha/posición (tomamos la primera posición para leer el valor)
                    # si la posición existe en serie_interp, usamos iloc para tomar el valor en la posición encontrada
                    # (si hay varias filas con mismo timestamp, asignaremos el mismo valor a todas ellas)
                    val = None
                    try:
                        val = serie_interp.iloc[positions[0]]
                    except Exception:
                        # fallback: intentar seleccionar por etiqueta (puede devolver array si hay duplicados)
                        try:
                            val = serie_interp.loc[ts]
                            # si loc devolvió una Serie (duplicados), tomar el primer elemento
                            if isinstance(val, (pd.Series, np.ndarray, list)):
                                val = val.iloc[0] if isinstance(val, pd.Series) else val[0]
                        except Exception:
                            val = np.nan
                    # asignar a todas las posiciones correspondientes
                    for p in positions:
                        df_imp.iat[p, col_loc] = val

            elif L < MEDIUM_GAP_MAX:
                # imputación por estadístico horario si existe
                if col in hourly_stats_dict:
                    hourly_series = hourly_stats_dict[col]  # Serie con índice 0..23
                    for ts in idx:
                        positions = np.where(index_values == np.datetime64(ts))[0]
                        if positions.size == 0:
                            # timestamp no presente en el DataFrame (ignorar)
                            continue
                        # calcular hora
                        try:
                            hour = pd.to_datetime(ts).hour
                        except Exception:
                            # si no se puede convertir, saltar
                            continue
                        # obtener valor horario (si no existe, NaN)
                        try:
                            # hourly_series es una Series con índices 0..23
                            val = hourly_series.loc[hour]
                        except Exception:
                            # si falla, intentar get
                            val = hourly_series.get(hour, np.nan) if hasattr(hourly_series, 'get') else np.nan
                        # asignar a todas las posiciones correspondientes
                        for p in positions:
                            df_imp.iat[p, col_loc] = val
                else:
                    # no hay estadístico horario para esta columna -> dejamos NaN para KNN
                    pass
            else:
                # gap largo: dejamos NaN para KNN
                pass

    return df_imp

def apply_knn_imputation(df, num_cols, extra_cols=None):
    """
    Aplica KNNImputer a las columnas num_cols (solo las que existan en df),
    usando extra_cols como predictores (deben ser numéricos).
    Retorna df con las columnas imputadas.
    """
    cols_to_impute = [c for c in num_cols if c in df.columns]
    if not cols_to_impute:
        return df

    if extra_cols:
        extra_cols = [c for c in extra_cols if c in df.columns]
        all_cols = cols_to_impute + extra_cols
    else:
        all_cols = cols_to_impute

    sub = df[all_cols].copy()
    for c in sub.columns:
        sub[c] = pd.to_numeric(sub[c], errors='coerce')

    if extra_cols:
        for col in extra_cols:
            if sub[col].isna().any():
                sub[col].fillna(-999, inplace=True)

    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS)
    try:
        imputed_array = imputer.fit_transform(sub)
    except Exception as e:
        print("  ERROR en KNNImputer:", e)
        print("  Intentando imputación de fallback por media de columna.")
        for c in cols_to_impute:
            mean_val = sub[c].mean(skipna=True)
            sub[c].fillna(mean_val if not pd.isna(mean_val) else 0.0, inplace=True)
        imputed_array = sub.values

    df_imp = df.copy()
    df_imp.loc[:, cols_to_impute] = imputed_array[:, :len(cols_to_impute)]
    return df_imp

def fill_remaining_nans_with_mean(df, cols):
    """
    Rellena los NaN que aún puedan quedar en las columnas especificadas con la media de cada columna.
    Si una columna es completamente NaN, se rellena con 0 (último recurso).
    """
    for col in cols:
        if col in df.columns and df[col].isna().any():
            mean_val = pd.to_numeric(df[col], errors='coerce').mean(skipna=True)
            if pd.isna(mean_val):
                mean_val = 0.0
            df[col] = df[col].fillna(mean_val)
    return df

def ensure_no_nans(df, cols):
    """
    Verifica que no queden NaN en cols; si quedan, rellena con 0 e imprime advertencia.
    """
    for col in cols:
        if col in df.columns and df[col].isna().any():
            n_nans = int(df[col].isna().sum())
            print(f"  ADVERTENCIA: {n_nans} NaN residuales en '{col}' rellenados con 0.")
            df[col] = df[col].fillna(0)
    return df

# ============================================================================
# PROCESAMIENTO POR TRANSECTO (MEJORADO)
# ============================================================================

def process_by_transect():
    print("\n=== Imputación por transecto (agrupando estaciones) ===")

    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    transect_dict = {}

    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        df = load_station_data(filepath)

        if 'O3' in df.columns and 'O3_for_impute' not in df.columns:
            df['O3_for_impute'] = pd.to_numeric(df['O3'], errors='coerce')

        if 'Estacion' not in df.columns:
            df['Estacion'] = station_name
        else:
            df['Estacion'] = df['Estacion'].fillna(station_name)

        if 'Transecto' not in df.columns:
            print(f"  Advertencia: {filepath.name} no tiene columna 'Transecto'. Se omite.")
            continue
        transect_names = df['Transecto'].dropna().unique()
        if len(transect_names) == 0:
            print(f"  Advertencia: {filepath.name} no tiene valores en Transecto. Se omite.")
            continue
        transect = transect_names[0]
        df['Transecto'] = df['Transecto'].fillna(transect)

        transect_clean = str(transect).replace(" ", "_")

        if transect_clean not in transect_dict:
            transect_dict[transect_clean] = []
        transect_dict[transect_clean].append((station_name, df))

    for transect_clean, station_list in transect_dict.items():
        print(f"\nProcesando transecto: {transect_clean}")

        df_all = pd.concat([df for _, df in station_list], axis=0, sort=False)
        df_all = df_all.sort_index()

        hourly_stats = {}
        for col in NUM_COLS:
            if col not in df_all.columns:
                continue
            s = pd.to_numeric(df_all[col], errors='coerce').dropna()
            hourly_stats[col] = compute_hourly_stats(s, SEASONAL_STAT)

        df_imputed_list = []
        for station_name, df_station in station_list:
            df_station['Estacion'] = df_station['Estacion'].fillna(station_name)
            if 'Transecto' in df_station.columns:
                df_station['Transecto'] = df_station['Transecto'].fillna(transect.replace("_", " "))
            for col in NUM_COLS:
                if col in df_station.columns:
                    df_station[col] = pd.to_numeric(df_station[col], errors='coerce')

            df_station_imp = impute_dataframe(df_station, hourly_stats)
            df_imputed_list.append(df_station_imp)

        df_concat = pd.concat(df_imputed_list, axis=0, sort=False)
        df_concat = df_concat.sort_index()

        # Reemplazamos fillna(method=...) por ffill().bfill()
        if 'Estacion' in df_concat.columns:
            df_concat['Estacion'] = df_concat['Estacion'].ffill().bfill().fillna('unknown_station')
        if 'Transecto' in df_concat.columns:
            df_concat['Transecto'] = df_concat['Transecto'].ffill().bfill().fillna(transect_clean.replace("_", " "))

        df_concat['station_code'] = pd.factorize(df_concat['Estacion'])[0]
        extra_cols = ['station_code']

        df_concat = apply_knn_imputation(df_concat, NUM_COLS, extra_cols=extra_cols)
        df_concat = fill_remaining_nans_with_mean(df_concat, NUM_COLS)
        df_concat = ensure_no_nans(df_concat, NUM_COLS)

        if 'station_code' in df_concat.columns:
            df_concat.drop(columns=['station_code'], inplace=True)

        if 'O3_for_impute' in df_concat.columns:
            df_concat['O3'] = df_concat['O3_for_impute']
            df_concat.drop(columns=['O3_for_impute'], inplace=True)

        out_path = os.path.join(OUTPUT_BY_TRANSECT, f"{transect_clean}.csv")
        df_concat.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path} (shape: {df_concat.shape})")

# ============================================================================
# PROCESAMIENTO GLOBAL (MEJORADO)
# ============================================================================

def process_global():
    print("\n=== Imputación global (todas las estaciones) ===")
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    dfs = []
    station_names = []
    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        station_names.append(station_name)
        df = load_station_data(filepath)

        # Asegurar columnas necesarias y tipos numéricos
        if 'O3' in df.columns and 'O3_for_impute' not in df.columns:
            df['O3_for_impute'] = pd.to_numeric(df['O3'], errors='coerce')
        if 'Estacion' not in df.columns:
            df['Estacion'] = station_name
        else:
            df['Estacion'] = df['Estacion'].fillna(station_name)
        if 'Transecto' in df.columns:
            tran_names = df['Transecto'].dropna().unique()
            if len(tran_names) > 0:
                df['Transecto'] = df['Transecto'].fillna(tran_names[0])

        # Convertir a numérico todas las columnas de interés de una vez
        for col in NUM_COLS:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        df['station_id'] = station_name
        dfs.append(df)

    # Concatenar todas las estaciones
    df_all = pd.concat(dfs, axis=0, sort=False)
    df_all = df_all.sort_index()

    # Calcular estadísticos horarios globales (una sola vez)
    global_hourly_stats = {}
    for col in NUM_COLS:
        if col not in df_all.columns:
            continue
        s = df_all[col].dropna()
        global_hourly_stats[col] = compute_hourly_stats(s, SEASONAL_STAT)

    # Imputar cada estación por separado y guardar en lista
    imputed_list = []
    for station in station_names:
        mask = df_all['station_id'] == station
        df_station = df_all.loc[mask].copy()  # copia necesaria para no alterar el original
        df_station_imp = impute_dataframe(df_station, global_hourly_stats)
        imputed_list.append(df_station_imp)

    # Concatenar resultados imputados
    df_all_imp = pd.concat(imputed_list, axis=0).sort_index()

    # KNN global con código de estación
    df_all_imp['station_code'] = pd.factorize(df_all_imp['station_id'])[0]
    extra_cols = ['station_code']
    df_all_imp = apply_knn_imputation(df_all_imp, NUM_COLS, extra_cols=extra_cols)
    df_all_imp = fill_remaining_nans_with_mean(df_all_imp, NUM_COLS)
    df_all_imp = ensure_no_nans(df_all_imp, NUM_COLS)

    # Limpiar columnas auxiliares y renombrar O3
    if 'station_code' in df_all_imp.columns:
        df_all_imp.drop(columns=['station_code'], inplace=True)
    if 'O3_for_impute' in df_all_imp.columns:
        df_all_imp['O3'] = df_all_imp['O3_for_impute']
        df_all_imp.drop(columns=['O3_for_impute'], inplace=True)

    # Guardar cada estación por separado
    for station in station_names:
        df_station = df_all_imp[df_all_imp['station_id'] == station].copy()
        if 'station_id' in df_station.columns:
            df_station.drop(columns=['station_id'], inplace=True)
        # Rellenar posibles NaN residuales en categóricas
        if 'Estacion' in df_station.columns and df_station['Estacion'].isna().any():
            df_station['Estacion'] = df_station['Estacion'].ffill().bfill().fillna(station)
        if 'Transecto' in df_station.columns and df_station['Transecto'].isna().any():
            df_station['Transecto'] = df_station['Transecto'].ffill().bfill().fillna('unknown_transect')

        out_path = os.path.join(OUTPUT_GLOBAL, f"{station}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path}")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("Iniciando imputación de datos (por transecto y global)...")
    process_by_transect()
    process_global()
    print("\nProceso completado. Revise las carpetas:")
    print(f"  - {OUTPUT_BY_TRANSECT}")
    print(f"  - {OUTPUT_GLOBAL}")

# Mas Optimizado

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # Imputacion valores NAN y OUTLIERS segun rango temporal con metodo por Transecto y metodo General
# Versión corregida: maneja índices duplicados al imputar gaps y evita errores de reindexación.
# Cambios principales:
#  - Lectura de CSV con low_memory=False para evitar DtypeWarning.
#  - Imputación por horas y asignaciones realizadas por posición (i.e. iat/iloc) para soportar índices duplicados.
#  - Mensajes de depuración mínimos para gaps no encontrados en el índice.
#
# Entrada: Archivos *_outliers.csv en INPUT_DIR
# Salida:
#  - imputed_by_transect/: archivos Transecto_1.csv, Transecto_2.csv, ...
#  - imputed_global/     : archivos por estación

import os
import numpy as np
import pandas as pd
from pathlib import Path
import concurrent.futures
from sklearn.impute import KNNImputer

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/usu/snsaetor/Documents/GitHub/TFGFinal/Datos_TFG_outliers/")
INPUT_DIR = BASE_DIR  # donde están los *_outliers.csv

OUTPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
OUTPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_TRANSECT, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

# Umbrales (horas)
SHORT_GAP_MAX = 6
MEDIUM_GAP_MAX = 72
LONG_GAP_MIN = 72

# Método para imputación estacional ('mean' o 'median')
SEASONAL_STAT = 'mean'

# Columnas numéricas a imputar (incluye tanto O3_for_impute como O3 por seguridad)
NUM_COLS = [
    'NO', 'NO2', 'NOx', 'O3_for_impute', 'O3',
    'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Dist.', 'Angulo'
]

# Para KNN
KNN_NEIGHBORS = 5

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def load_station_data(filepath):
    """Carga un CSV con índice datetime."""
    # low_memory=False para evitar DtypeWarning en columnas mezcladas
    df = pd.read_csv(filepath, index_col=0, parse_dates=True, low_memory=False)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    return df

def get_gap_lengths(series):
    """
    Devuelve un dict {tuple(indice): longitud} para cada grupo consecutivo de NaN en 'series'.
    Cada key es una tupla de timestamps (puede contener timestamps repetidos si existían en el índice).
    """
    mask = series.isna()
    if mask.empty:
        return {}
    groups = mask.ne(mask.shift()).cumsum()
    gap_groups = {}
    # iterar sobre grupos donde mask == True
    for g in groups[mask].unique():
        idx = groups[groups == g].index
        gap_groups[tuple(idx)] = len(idx)
    return gap_groups

def compute_hourly_stats(series, stat='mean'):
    """
    Calcula el estadístico horario (media o mediana) y asegura que existan valores para todas las horas (0-23).
    Si no hay datos para una hora, se rellena con el valor más cercano (ffill/bfill).
    Si la serie está vacía, retorna Serie de NaN.
    """
    if series is None or series.empty:
        return pd.Series(np.nan, index=range(24))
    if stat == 'mean':
        hourly = series.groupby(series.index.hour).mean()
    else:
        hourly = series.groupby(series.index.hour).median()
    hourly = hourly.reindex(range(24))
    hourly = hourly.ffill().bfill()
    return hourly

def impute_dataframe(df, hourly_stats_dict):
    """
    Imputa un DataFrame de una estación:
      - Gaps cortos (<= SHORT_GAP_MAX): interpolación lineal.
      - Gaps medios (< MEDIUM_GAP_MAX): imputación por estadístico horario.
      - Gaps largos (>= MEDIUM_GAP_MAX): se dejan para KNN.
    Esta versión maneja índices duplicados asignando por posición.
    """
    df_imp = df.copy()
    # Precalcular un array del índice para comparaciones rápidas
    index_values = df_imp.index.values

    for col in NUM_COLS:
        if col not in df_imp.columns:
            continue
        # Forzar tipo numérico
        df_imp[col] = pd.to_numeric(df_imp[col], errors='coerce')
        serie = df_imp[col]
        if serie.isna().sum() == 0:
            continue

        gaps = get_gap_lengths(serie)
        if not gaps:
            continue

        # ubicación de la columna en df_imp (entera) para uso con iat
        col_loc = df_imp.columns.get_loc(col)

        for idx_tuple, L in gaps.items():
            # idx contiene los timestamps (posiblemente repetidos) del gap
            idx = pd.Index(idx_tuple)

            if L <= SHORT_GAP_MAX:
                # interpolación lineal global en la serie (no solo en el gap)
                serie_interp = serie.interpolate(method='linear', limit_direction='both')
                # Para evitar reindex problemas con duplicados, asignamos por posiciones encontradas para cada timestamp
                for ts in idx:
                    # obtener posiciones donde el índice coincide exactamente con ts
                    positions = np.where(index_values == np.datetime64(ts))[0]
                    if positions.size == 0:
                        # timestamp no encontrado en el índice (raro), lo ignoramos
                        # (opcional: registrar)
                        # print(f"  Nota: timestamp {ts} del gap no se encontró en el índice de estación.")
                        continue
                    # valor interpolado en esa fecha/posición (tomamos la primera posición para leer el valor)
                    # si la posición existe en serie_interp, usamos iloc para tomar el valor en la posición encontrada
                    # (si hay varias filas con mismo timestamp, asignaremos el mismo valor a todas ellas)
                    val = None
                    try:
                        val = serie_interp.iloc[positions[0]]
                    except Exception:
                        # fallback: intentar seleccionar por etiqueta (puede devolver array si hay duplicados)
                        try:
                            val = serie_interp.loc[ts]
                            # si loc devolvió una Serie (duplicados), tomar el primer elemento
                            if isinstance(val, (pd.Series, np.ndarray, list)):
                                val = val.iloc[0] if isinstance(val, pd.Series) else val[0]
                        except Exception:
                            val = np.nan
                    # asignar a todas las posiciones correspondientes
                    for p in positions:
                        df_imp.iat[p, col_loc] = val

            elif L < MEDIUM_GAP_MAX:
                # imputación por estadístico horario si existe
                if col in hourly_stats_dict:
                    hourly_series = hourly_stats_dict[col]  # Serie con índice 0..23
                    for ts in idx:
                        positions = np.where(index_values == np.datetime64(ts))[0]
                        if positions.size == 0:
                            # timestamp no presente en el DataFrame (ignorar)
                            continue
                        # calcular hora
                        try:
                            hour = pd.to_datetime(ts).hour
                        except Exception:
                            # si no se puede convertir, saltar
                            continue
                        # obtener valor horario (si no existe, NaN)
                        try:
                            # hourly_series es una Series con índices 0..23
                            val = hourly_series.loc[hour]
                        except Exception:
                            # si falla, intentar get
                            val = hourly_series.get(hour, np.nan) if hasattr(hourly_series, 'get') else np.nan
                        # asignar a todas las posiciones correspondientes
                        for p in positions:
                            df_imp.iat[p, col_loc] = val
                else:
                    # no hay estadístico horario para esta columna -> dejamos NaN para KNN
                    pass
            else:
                # gap largo: dejamos NaN para KNN
                pass

    return df_imp

def apply_knn_imputation(df, num_cols, extra_cols=None):
    """
    Aplica KNNImputer a las columnas num_cols (solo las que existan en df),
    usando extra_cols como predictores (deben ser numéricos).
    Retorna df con las columnas imputadas.
    """
    cols_to_impute = [c for c in num_cols if c in df.columns]
    if not cols_to_impute:
        return df

    if extra_cols:
        extra_cols = [c for c in extra_cols if c in df.columns]
        all_cols = cols_to_impute + extra_cols
    else:
        all_cols = cols_to_impute

    sub = df[all_cols].copy()
    for c in sub.columns:
        sub[c] = pd.to_numeric(sub[c], errors='coerce')

    if extra_cols:
        for col in extra_cols:
            if sub[col].isna().any():
                sub[col].fillna(-999, inplace=True)

    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS)
    try:
        imputed_array = imputer.fit_transform(sub)
    except Exception as e:
        print("  ERROR en KNNImputer:", e)
        print("  Intentando imputación de fallback por media de columna.")
        for c in cols_to_impute:
            mean_val = sub[c].mean(skipna=True)
            sub[c].fillna(mean_val if not pd.isna(mean_val) else 0.0, inplace=True)
        imputed_array = sub.values

    df_imp = df.copy()
    df_imp.loc[:, cols_to_impute] = imputed_array[:, :len(cols_to_impute)]
    return df_imp

def fill_remaining_nans_with_mean(df, cols):
    """
    Rellena los NaN que aún puedan quedar en las columnas especificadas con la media de cada columna.
    Si una columna es completamente NaN, se rellena con 0 (último recurso).
    """
    for col in cols:
        if col in df.columns and df[col].isna().any():
            mean_val = pd.to_numeric(df[col], errors='coerce').mean(skipna=True)
            if pd.isna(mean_val):
                mean_val = 0.0
            df[col] = df[col].fillna(mean_val)
    return df

def ensure_no_nans(df, cols):
    """
    Verifica que no queden NaN en cols; si quedan, rellena con 0 e imprime advertencia.
    """
    for col in cols:
        if col in df.columns and df[col].isna().any():
            n_nans = int(df[col].isna().sum())
            print(f"  ADVERTENCIA: {n_nans} NaN residuales en '{col}' rellenados con 0.")
            df[col] = df[col].fillna(0)
    return df

# ============================================================================
# PROCESAMIENTO POR TRANSECTO (MEJORADO)
# ============================================================================

def process_by_transect():
    print("\n=== Imputación por transecto (agrupando estaciones) ===")

    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    transect_dict = {}

    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        df = load_station_data(filepath)

        if 'O3' in df.columns and 'O3_for_impute' not in df.columns:
            df['O3_for_impute'] = pd.to_numeric(df['O3'], errors='coerce')

        if 'Estacion' not in df.columns:
            df['Estacion'] = station_name
        else:
            df['Estacion'] = df['Estacion'].fillna(station_name)

        if 'Transecto' not in df.columns:
            print(f"  Advertencia: {filepath.name} no tiene columna 'Transecto'. Se omite.")
            continue
        transect_names = df['Transecto'].dropna().unique()
        if len(transect_names) == 0:
            print(f"  Advertencia: {filepath.name} no tiene valores en Transecto. Se omite.")
            continue
        transect = transect_names[0]
        df['Transecto'] = df['Transecto'].fillna(transect)

        transect_clean = str(transect).replace(" ", "_")

        if transect_clean not in transect_dict:
            transect_dict[transect_clean] = []
        transect_dict[transect_clean].append((station_name, df))

    for transect_clean, station_list in transect_dict.items():
        print(f"\nProcesando transecto: {transect_clean}")

        df_all = pd.concat([df for _, df in station_list], axis=0, sort=False)
        df_all = df_all.sort_index()

        hourly_stats = {}
        for col in NUM_COLS:
            if col not in df_all.columns:
                continue
            s = pd.to_numeric(df_all[col], errors='coerce').dropna()
            hourly_stats[col] = compute_hourly_stats(s, SEASONAL_STAT)

        df_imputed_list = []
        for station_name, df_station in station_list:
            df_station['Estacion'] = df_station['Estacion'].fillna(station_name)
            if 'Transecto' in df_station.columns:
                df_station['Transecto'] = df_station['Transecto'].fillna(transect.replace("_", " "))
            for col in NUM_COLS:
                if col in df_station.columns:
                    df_station[col] = pd.to_numeric(df_station[col], errors='coerce')

            df_station_imp = impute_dataframe(df_station, hourly_stats)
            df_imputed_list.append(df_station_imp)

        df_concat = pd.concat(df_imputed_list, axis=0, sort=False)
        df_concat = df_concat.sort_index()

        # Reemplazamos fillna(method=...) por ffill().bfill()
        if 'Estacion' in df_concat.columns:
            df_concat['Estacion'] = df_concat['Estacion'].ffill().bfill().fillna('unknown_station')
        if 'Transecto' in df_concat.columns:
            df_concat['Transecto'] = df_concat['Transecto'].ffill().bfill().fillna(transect_clean.replace("_", " "))

        df_concat['station_code'] = pd.factorize(df_concat['Estacion'])[0]
        extra_cols = ['station_code']

        df_concat = apply_knn_imputation(df_concat, NUM_COLS, extra_cols=extra_cols)
        df_concat = fill_remaining_nans_with_mean(df_concat, NUM_COLS)
        df_concat = ensure_no_nans(df_concat, NUM_COLS)

        if 'station_code' in df_concat.columns:
            df_concat.drop(columns=['station_code'], inplace=True)

        if 'O3_for_impute' in df_concat.columns:
            df_concat['O3'] = df_concat['O3_for_impute']
            df_concat.drop(columns=['O3_for_impute'], inplace=True)

        out_path = os.path.join(OUTPUT_BY_TRANSECT, f"{transect_clean}.csv")
        df_concat.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path} (shape: {df_concat.shape})")

# ============================================================================
# PROCESAMIENTO GLOBAL (MEJORADO)
# ============================================================================

def process_global():
    print("\n=== Imputación global (todas las estaciones) - Versión paralela ===")
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    # Cargar todas las estaciones y prepararlas
    stations_data = []  # lista de tuplas (station_name, df)
    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        df = load_station_data(filepath)

        # Asegurar columnas necesarias y tipos numéricos
        if 'O3' in df.columns and 'O3_for_impute' not in df.columns:
            df['O3_for_impute'] = pd.to_numeric(df['O3'], errors='coerce')
        if 'Estacion' not in df.columns:
            df['Estacion'] = station_name
        else:
            df['Estacion'] = df['Estacion'].fillna(station_name)
        if 'Transecto' in df.columns:
            tran_names = df['Transecto'].dropna().unique()
            if len(tran_names) > 0:
                df['Transecto'] = df['Transecto'].fillna(tran_names[0])

        # Convertir a numérico todas las columnas de interés de una vez
        for col in NUM_COLS:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        df['station_id'] = station_name
        stations_data.append((station_name, df))

    # Calcular estadísticos horarios globales (necesitamos todos los datos concatenados)
    df_all = pd.concat([df for _, df in stations_data], axis=0, sort=False)
    df_all = df_all.sort_index()

    global_hourly_stats = {}
    for col in NUM_COLS:
        if col not in df_all.columns:
            continue
        s = df_all[col].dropna()
        global_hourly_stats[col] = compute_hourly_stats(s, SEASONAL_STAT)

    # Función auxiliar para imputar una estación (debe estar al nivel superior para picklear)
    def impute_one_station(station_name, df):
        df_station_imp = impute_dataframe(df.copy(), global_hourly_stats)
        return station_name, df_station_imp

    # Paralelizar la imputación por estación
    num_workers = min(len(stations_data), os.cpu_count())
    imputed_list = []
    with concurrent.futures.ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(impute_one_station, name, df) for name, df in stations_data]
        for future in concurrent.futures.as_completed(futures):
            try:
                station_name, df_imp = future.result()
                imputed_list.append(df_imp)
                print(f"  Estación {station_name} imputada.")
            except Exception as e:
                print(f"  Error procesando estación: {e}")

    # Concatenar resultados imputados
    df_all_imp = pd.concat(imputed_list, axis=0).sort_index()

    # KNN global con código de estación
    df_all_imp['station_code'] = pd.factorize(df_all_imp['station_id'])[0]
    extra_cols = ['station_code']
    df_all_imp = apply_knn_imputation(df_all_imp, NUM_COLS, extra_cols=extra_cols)
    df_all_imp = fill_remaining_nans_with_mean(df_all_imp, NUM_COLS)
    df_all_imp = ensure_no_nans(df_all_imp, NUM_COLS)

    # Limpiar columnas auxiliares y renombrar O3
    if 'station_code' in df_all_imp.columns:
        df_all_imp.drop(columns=['station_code'], inplace=True)
    if 'O3_for_impute' in df_all_imp.columns:
        df_all_imp['O3'] = df_all_imp['O3_for_impute']
        df_all_imp.drop(columns=['O3_for_impute'], inplace=True)

    # Guardar cada estación por separado
    for station in station_names:
        df_station = df_all_imp[df_all_imp['station_id'] == station].copy()
        if 'station_id' in df_station.columns:
            df_station.drop(columns=['station_id'], inplace=True)
        # Rellenar posibles NaN residuales en categóricas
        if 'Estacion' in df_station.columns and df_station['Estacion'].isna().any():
            df_station['Estacion'] = df_station['Estacion'].ffill().bfill().fillna(station)
        if 'Transecto' in df_station.columns and df_station['Transecto'].isna().any():
            df_station['Transecto'] = df_station['Transecto'].ffill().bfill().fillna('unknown_transect')

        out_path = os.path.join(OUTPUT_GLOBAL, f"{station}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path}")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("Iniciando imputación de datos (por transecto y global)...")
    process_by_transect()
    process_global()
    print("\nProceso completado. Revise las carpetas:")
    print(f"  - {OUTPUT_BY_TRANSECT}")
    print(f"  - {OUTPUT_GLOBAL}")